# Original vs fixed behaviour

`implementation='original'` uses the original explainer (the `legacy` implementation of CtreeX);
`'fixed'` (the default) enables the CtreeX fixes documented in `CHANGELOG.md`,
each of which can also be switched individually.
The corrections of the dialogue and the intent mapping are always applied.

In [1]:
import random
from glassbox.dialogue import DialogueManager
from glassbox.model import GlassBoxModel

def conversation(implementation, channel='chat', **kw):
    random.seed(2018)
    manager = DialogueManager(GlassBoxModel(implementation=implementation),
                              implementation=implementation,
                              **kw)
    script = [('hello', {}),
              ('apply', {'persona': '03'}),
              ('myself', {}),
              ('why', {}),
              ('fairness', {}),
              ('action', {})]
    return [manager.process_intent(i, a, channel=channel) for i, a in script]

for original, fixed in zip(conversation('original'), conversation('fixed')):
    print('ORIGINAL:', original)
    print('FIXED:   ', fixed, '\n')

ORIGINAL: Hello. Welcome to Robo Bank. If you would like to apply for a loan please provide your name or application number. Alternatively please scan your application card.
FIXED:    Hello. Welcome to Robo Bank. If you would like to apply for a loan please provide your name or application number. Alternatively please scan your application card. 

ORIGINAL: Thank you for your application, Ruth. Please bear with me a moment while I process your application. Your loan application has been processed. Would you like to know the outcome?
FIXED:    Thank you for your application, Ruth. Please bear with me a moment while I process your application. Your loan application has been processed. Would you like to know the outcome? 

ORIGINAL: Dear Ruth, at the time of application: you have requested your loan for 15 months and you have asked for 3959 pounds and you have had overdraft on your current account and you have had less than 100 pounds on your savings account and you have not had other loa

## Explainer fixes

The explainer is [CtreeX][ctreex];
`implementation='original'` uses its `legacy` implementation.
Of its fixes, only `complete_categorical_features` changes Glass-Box:
the tree splits on some categories of the job type (A17) and the housing type (A15) only, so the legacy implementation describes some leaves incorrectly, e.g., applicants end up at a non-zero distance from their own leaf.

[ctreex]: https://github.com/So-Cool/CtreeX

In [2]:
import numpy as np
from mlaas.data_loader import DataHolder
from glassbox import data

dh = data.load_german_credit()
for implementation in ('original', 'fixed'):
    explainer = GlassBoxModel(implementation=implementation).explainer
    ids = explainer.leaf_meta_features.leaf_ids.tolist()
    off = sum(
        explainer.explain_instance(x).leaf_distances[ids.index(explainer.predict_leaf(x))]
        != 0 for x in dh.data_onehot)
    print('{:8s} applicants not at distance 0 from their own leaf: '
          '{} of {}'.format(implementation, off, len(dh.data)))

Using given data set (loaded from *mlaas.data.german_credit_simplified* module)


original applicants not at distance 0 from their own leaf: 58 of 1000


fixed    applicants not at distance 0 from their own leaf: 0 of 1000


The answers of the personas with and without this fix (all the other fixes enabled):

In [3]:
for pid in ['01', '06']:
    answers = []
    for fixes in (None, {'complete_categorical_features': False}):
        random.seed(2018)
        manager = DialogueManager(GlassBoxModel(fixes=fixes))
        manager.process_intent('hello')
        manager.process_intent('apply', {'persona': pid})
        answers.append([manager.process_intent('why') for _ in range(4)])
    for with_fix, without_fix in zip(*answers):
        if with_fix != without_fix:
            print('WITH:   ', with_fix)
            print('WITHOUT:', without_fix, '\n')

WITH:    I do not have any ongoing enquiry. Please give me a moment to open one for you. If you had asked for at most 663 or between 839 and 883 or at least 2774 pounds, rather than 2319 pounds, your loan would be accepted.
WITHOUT: I do not have any ongoing enquiry. Please give me a moment to open one for you. If you had asked for at most 663 or between 839 and 883 pounds, rather than 2319 pounds, your loan would be accepted. 

WITH:    Your loan would be accepted if you had requested your loan for at most 7 or between 12 and 15 or between 23 and 43 months, rather than 21 months.
WITHOUT: Your loan would be accepted if you had requested your loan for at most 7 months, rather than 21 months. 

WITH:    Had you been male (divorced or separated) or male (single) or male (married or widowed), instead of being female (divorced or separated or married), your loan would be declined.
WITHOUT: Had you been male (divorced or separated) or male (married or widowed) or male (single), instead of b